# Spark to PostgreSQL Integration

This notebook demonstrates writing Spark DataFrames to PostgreSQL.

**Prerequisites:**
- Start containers with: `docker-compose --profile with-db up`
- PostgreSQL will be available at `jupyter-postgres:5432`

## 1. Create SparkSession with JDBC Driver

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
from datetime import date

# Create SparkSession with PostgreSQL JDBC driver
spark = SparkSession.builder \
    .appName("Spark PostgreSQL Integration") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.1") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print("SparkSession created with PostgreSQL JDBC driver!")

## 2. Define PostgreSQL Connection Properties

In [ ]:
# PostgreSQL connection settings
jdbc_url = "jdbc:postgresql://jupyter-postgres:5432/dataeng"

connection_properties = {
    "user": "jupyter",
    "password": "jupyter_password",
    "driver": "org.postgresql.Driver"
}

print(f"JDBC URL: {jdbc_url}")
print("Connection properties configured.")

## 3. Create Sample DataFrames

In [ ]:
# Employees DataFrame
employees_data = [
    (1, "Alice", "Engineering", 75000.00, date(2020, 3, 15)),
    (2, "Bob", "Engineering", 80000.00, date(2019, 7, 22)),
    (3, "Charlie", "Sales", 65000.00, date(2021, 1, 10)),
    (4, "Diana", "Sales", 70000.00, date(2020, 11, 5)),
    (5, "Eve", "Marketing", 60000.00, date(2022, 4, 18)),
    (6, "Frank", "Engineering", 90000.00, date(2018, 9, 1)),
    (7, "Grace", "Marketing", 55000.00, date(2023, 2, 28)),
    (8, "Henry", "Sales", 72000.00, date(2021, 8, 14)),
]

employees_schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("department", StringType(), False),
    StructField("salary", DoubleType(), False),
    StructField("hire_date", DateType(), False)
])

employees_df = spark.createDataFrame(employees_data, employees_schema)

print("Employees DataFrame:")
employees_df.show()
employees_df.printSchema()

In [ ]:
# Sales DataFrame
sales_data = [
    (1, "P001", "North", 1500.00, date(2024, 1, 15)),
    (2, "P002", "South", 2300.00, date(2024, 1, 16)),
    (3, "P001", "East", 1800.00, date(2024, 1, 17)),
    (4, "P003", "North", 950.00, date(2024, 1, 18)),
    (5, "P002", "West", 2100.00, date(2024, 1, 19)),
    (6, "P001", "South", 1650.00, date(2024, 1, 20)),
    (7, "P003", "East", 1100.00, date(2024, 1, 21)),
    (8, "P002", "North", 2500.00, date(2024, 1, 22)),
]

sales_schema = StructType([
    StructField("sale_id", IntegerType(), False),
    StructField("product_id", StringType(), False),
    StructField("region", StringType(), False),
    StructField("amount", DoubleType(), False),
    StructField("sale_date", DateType(), False)
])

sales_df = spark.createDataFrame(sales_data, sales_schema)

print("Sales DataFrame:")
sales_df.show()

## 4. Write DataFrames to PostgreSQL

In [ ]:
# Write employees table
# mode options: "overwrite", "append", "ignore", "error"
print("Writing employees table to PostgreSQL...")

employees_df.write.jdbc(
    url=jdbc_url,
    table="employees",
    mode="overwrite",
    properties=connection_properties
)

print("Employees table written successfully!")

In [ ]:
# Write sales table
print("Writing sales table to PostgreSQL...")

sales_df.write.jdbc(
    url=jdbc_url,
    table="sales",
    mode="overwrite",
    properties=connection_properties
)

print("Sales table written successfully!")

## 5. Read Data Back from PostgreSQL

In [ ]:
# Read employees table back
print("Reading employees table from PostgreSQL...")

employees_from_pg = spark.read.jdbc(
    url=jdbc_url,
    table="employees",
    properties=connection_properties
)

print(f"Rows retrieved: {employees_from_pg.count()}")
employees_from_pg.show()

In [ ]:
# Read sales table back
print("Reading sales table from PostgreSQL...")

sales_from_pg = spark.read.jdbc(
    url=jdbc_url,
    table="sales",
    properties=connection_properties
)

print(f"Rows retrieved: {sales_from_pg.count()}")
sales_from_pg.show()

## 6. Run SQL Query via JDBC

In [ ]:
# Execute a custom SQL query against PostgreSQL
query = "(SELECT department, COUNT(*) as emp_count, AVG(salary) as avg_salary FROM employees GROUP BY department) AS dept_stats"

print("Running aggregation query on PostgreSQL...")

dept_stats = spark.read.jdbc(
    url=jdbc_url,
    table=query,
    properties=connection_properties
)

print("Department Statistics (from PostgreSQL):")
dept_stats.show()

## 7. Append New Data

In [ ]:
# Create new employees to append
new_employees_data = [
    (9, "Ivy", "Engineering", 85000.00, date(2024, 1, 8)),
    (10, "Jack", "Sales", 68000.00, date(2024, 1, 15)),
]

new_employees_df = spark.createDataFrame(new_employees_data, employees_schema)

print("New employees to append:")
new_employees_df.show()

# Append to existing table
new_employees_df.write.jdbc(
    url=jdbc_url,
    table="employees",
    mode="append",
    properties=connection_properties
)

print("New employees appended!")

In [ ]:
# Verify the append
print("All employees after append:")

all_employees = spark.read.jdbc(
    url=jdbc_url,
    table="employees",
    properties=connection_properties
)

print(f"Total rows: {all_employees.count()}")
all_employees.orderBy("id").show()

## 8. Cleanup

In [ ]:
spark.stop()
print("SparkSession stopped.")
print("\nSpark to PostgreSQL integration test completed!")